In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

spark = SparkSession.builder \
            .appName("Spark Batch Layer") \
            .master("local[*]") \
            .getOrCreate()

path = "/data/raw/prices"

25/12/16 22:17:03 WARN Utils: Your hostname, node1 resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
25/12/16 22:17:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/16 22:17:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
df = spark.read.parquet(path)

In [12]:
df = spark.read.parquet(path)

# 2. Sprawdźmy co tam siedzi
print("Schemat danych (mamy też kolumny partycji na końcu)")
df.printSchema()

print(f"Całkowita liczba rekordów w HDFS: {df.count()}")

report = df.groupBy("currency", "year", "month", "day").agg(avg("price").alias("avg_price"), count("price").alias("count")).orderBy("year", "month", "day", "currency")
report.show()



Schemat danych (mamy też kolumny partycji na końcu)
root
 |-- currency: string (nullable = true)
 |-- price: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)

Całkowita liczba rekordów w HDFS: 26361


[Stage 39:===================================>                  (132 + 2) / 200]

+--------+----+-----+---+-------------------+-----+
|currency|year|month|day|          avg_price|count|
+--------+----+-----+---+-------------------+-----+
| bitcoin|2025|   12| 15|  90032.62504749002| 1409|
|dogecoin|2025|   12| 15|0.19978763014171153| 1409|
|ethereum|2025|   12| 15|  2500.447573776003| 1409|
| bitcoin|2025|   12| 16|  90043.46243566377| 7378|
|dogecoin|2025|   12| 16| 0.1999447617080068| 7378|
|ethereum|2025|   12| 16|  2498.198194960236| 7378|
+--------+----+-----+---+-------------------+-----+



In [13]:
report.write.mode("overwrite").partitionBy("year", "month").saveAsTable("crypto_daily_reports")
print("Raporcik zapisany w Hive!")

Raporcik zapisany w Hive!
